## Imports & Setup

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,

    EPOCHS_ELECTRA     = 2,
    BS_ELECTRA         = 4,
    GRAD_ACCUM_ELECTRA = 4,        
    MAX_LEN_ELECTRA    = 384,      
    LR_ELECTRA         = 8e-6,    
    WARMUP_RATIO       = 0.1,     
 
    MARGIN        = 0.5,
    USE_BF16      = True,          
    
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
    ARTIFACT_DIR  = '/kaggle/working/artifacts',
    ELECTRA_MODEL = 'google/electra-large-discriminator'
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)
OPTS = ['A', 'B', 'C', 'D', 'E']

In [2]:
import os, sys, json, math, random, time, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler

from sklearn.metrics import f1_score, accuracy_score


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(CFG.get('SEED', 42))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

if CFG.get('USE_BF16', True) and DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    AMP_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    AMP_DTYPE = torch.float16
else:
    AMP_DTYPE = torch.float32
USE_SCALER = (AMP_DTYPE == torch.float16)
print(f"AMP dtype: {AMP_DTYPE} | GradScaler: {USE_SCALER}")

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch.set_float32_matmul_precision('high')



def autocast(force_fp32=False):
    if force_fp32:
        return _autocast(device_type='cuda', dtype=torch.float32, enabled=False)
    return _autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda'))


def make_scaler(force_fp32=False):

    return _GradScaler('cuda', enabled=(USE_SCALER and not force_fp32))

## Load Data

In [8]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
print(f"train: {train.shape}, test: {test.shape}")


train: (2000, 8), test: (500, 7)


## Normalize + GroupKFold Assignment

In [9]:
_BOILERPLATE = [
    r"pick the best possible answer:?",
    r"\bcarefully\b\.?",
    r"among the listed options\.?",
    r"choose the (correct|best) (option|answer|response)\.?",
    r"select the (best|correct|most appropriate) (option|answer|response)\.?",
    r"determine the correct option:?",
    r"identify the correct statement:?",
    r"based on the given context\.?",
    r"which of the following( statements)?( is true)?",
]
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = str(text)
    for pat in _BOILERPLATE:
        s = re.sub(pat, ' ', s, flags=re.IGNORECASE)
    s = _WS.sub(' ', s).strip()
    return s

for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)

print("Normalized. Sample prompt_n:")
print(" ", train['prompt_n'].iloc[0][:200])

gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.loc[train.index[va_idx], 'fold'] = fold_id

assert (train['fold'] >= 0).all(), 'Some rows did not get a fold assignment'
print(f"\nFold distribution:\n{train['fold'].value_counts().sort_index().to_string()}")

_dup_check = train.groupby('prompt_n')['fold'].nunique()
assert _dup_check.max() == 1, f"Leak: {(_dup_check > 1).sum()} prompts span multiple folds"
print(f"Verified: no prompt spans multiple folds.")

CONTEXT_MAP = {}


Normalized. Sample prompt_n:
  What is Martin Heidegger's view on the relationship between time and human existence?

Fold distribution:
fold
0    1000
1    1000
Verified: no prompt spans multiple folds.


## Metrics & Loss 

In [2]:

def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = order[i, :3]
        if y_idx[i] in top3:
            total += 1.0 / (list(top3).index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    pred_top1 = scores_2d.argmax(axis=1)
    return dict(
        map3 = map_at_3(scores_2d, y_idx),
        acc  = accuracy_score(y_idx, pred_top1),
        f1   = f1_score(y_idx, pred_top1, average='macro'),
    )


def pairwise_margin_loss(scores, correct, margin=0.5):
    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss); mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)


print("Metrics and loss defined.")

Metrics and loss defined.


## W&B Setup

In [4]:
import os
try:
    import wandb
except ImportError:
    wandb = None


USE_WANDB     = True
RUN_VERSION   = 4
RUN_NOTES     = "electra_run_4, n_folds=2, epoch=2"
WANDB_PROJECT = "24f1000134-t22026"

if USE_WANDB and wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("wandb_api"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}); set Kaggle secret 'wandb_api' to enable.")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",          
        job_type="train",
        notes=RUN_NOTES,                           
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION,
                "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc
wandb: Currently logged in as: 24f1000134 (24f1000134-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B connected.


### Uploading Model to Kagglehub

In [1]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_bddc34ec5bf3fe0fe1b234e7aab9ffbc"
import kagglehub

KAGGLE_USERNAME = "kamal134134"
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    "electra": f"{KAGGLE_USERNAME}/smart-mcq-electra/{HUB_FRAMEWORK}/updated_final_try"
}

def push_model(model_tag, folder, notes=""):
    if not PUSH_TO_HUB or kagglehub is None:
        print(f"[hub] upload skipped for '{model_tag}' (PUSH_TO_HUB=False).")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"[hub] uploading '{model_tag}' -> {handle}")
    kagglehub.model_upload(handle, folder, version_notes=note)
    print("Done. Inference notebooks can now kagglehub.")

Done. Inference notebooks can now kagglehub.


## HF Auth


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token loaded.")
else:
    print("No HF token found — public models will still download, gated ones won't.")

In [6]:
print("Downloading electra-large-discriminator tokenizer...")
t = time.time()
_tok = AutoTokenizer.from_pretrained(CFG['ELECTRA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  tokenizer ({time.time()-t:.0f}s)")

print("Downloading electra-large-discriminator model weights (~1.3 GB)...")
t = time.time()
_mdl = AutoModel.from_pretrained(CFG['ELECTRA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  model ({time.time()-t:.0f}s)")

del _mdl, _tok; gc.collect(); torch.cuda.empty_cache()
print("Models cached.")


  tokenizer (0s)


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 27606.71it/s]
[transformers] ElectraModel LOAD REPORT from: google/electra-large-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  model (2s)
Models cached.


## Model and Dataset 

In [ ]:
class ElectraScorer(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.backbone.gradient_checkpointing_enable() 
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(h, h // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(h // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls).squeeze(-1)


class ElectraDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, context_map=None,
                 shuffle_opts=False, has_label=True, seed=0):
        self.df       = df.reset_index(drop=True)
        self.tok      = tokenizer
        self.max_len  = max_len
        self.ctx      = context_map or {}
        self.shuffle  = shuffle_opts
        self.has_label = has_label
        self.rng      = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        prompt = r['prompt_n']
        if r['id'] in self.ctx:
            prompt = self.ctx[r['id']] + ' ' + prompt
        options = [r[f'{o}_n'] for o in OPTS]
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            options = [options[i] for i in perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])
        enc = self.tok([prompt]*5, options, truncation=True,
                       padding='max_length', max_length=self.max_len,
                       return_tensors='pt')
        return dict(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            correct=torch.tensor(correct, dtype=torch.long),
            id=int(r['id']),
        )


## Training Function

In [11]:
def train_electra_cv(train_df, test_df, n_folds=None, save_dir=None):
    if n_folds  is None: n_folds  = CFG['N_FOLDS']
    if save_dir is None: save_dir = CFG['ARTIFACT_DIR']
    os.makedirs(save_dir, exist_ok=True)

    MODEL_TAG = 'electra'

    oof_scores  = np.zeros((len(train_df), 5), dtype=np.float32)
    test_scores = np.zeros((len(test_df),  5), dtype=np.float32)
    y_idx       = np.array([OPTS.index(a) for a in train_df['answer']])

    tokenizer = AutoTokenizer.from_pretrained(CFG['ELECTRA_MODEL'])

    assert 'fold' in train_df.columns, "train_df must have a 'fold' column"

    for fold in range(n_folds):
  
        run = wandb_init(MODEL_TAG, fold)

        va_idx = np.where(train_df['fold'].values == fold)[0]
        tr_idx = np.where(train_df['fold'].values != fold)[0]
        print(f'\n[ELECTRA] === fold {fold} (train={len(tr_idx)}, val={len(va_idx)}) ===', flush=True)

        tr_ds = ElectraDataset(train_df.iloc[tr_idx], tokenizer, CFG['MAX_LEN_ELE'],
                               CONTEXT_MAP, shuffle_opts=True,  seed=fold)
        va_ds = ElectraDataset(train_df.iloc[va_idx], tokenizer, CFG['MAX_LEN_ELE'],
                               CONTEXT_MAP, shuffle_opts=False)
        te_ds = ElectraDataset(test_df,              tokenizer, CFG['MAX_LEN_ELE'],
                               CONTEXT_MAP, shuffle_opts=False, has_label=False)

        tr_dl = DataLoader(tr_ds, batch_size=CFG['BS_ELE'], shuffle=True,
                           num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, batch_size=CFG['BS_ELE'], shuffle=False,
                           num_workers=0, pin_memory=True)
        te_dl = DataLoader(te_ds, batch_size=CFG['BS_ELE'], shuffle=False,
                           num_workers=0, pin_memory=True)

      
        model  = ElectraScorer(CFG['ELECTRA_MODEL']).float().to(DEVICE)
        optim  = AdamW(model.parameters(), lr=CFG['LR_ELE'], weight_decay=0.01)
        steps  = CFG['EPOCHS_ELE'] * len(tr_dl) // CFG['GRAD_ACCUM_ELE']
        warmup = int(CFG['WARMUP_RATIO'] * steps)
        sched  = get_cosine_schedule_with_warmup(optim, num_warmup_steps=warmup,
                                                 num_training_steps=max(steps, 1))
        scaler = make_scaler()
        accum  = CFG['GRAD_ACCUM_ELE']

        best_val_map3   = -1
        best_va_scores  = None
        best_te_scores  = None
        best_state_dict = None

        for epoch in range(CFG['EPOCHS_ELE']):
            t_epoch    = time.time()
            model.train()
            train_loss = 0.0; n_train = 0
            optim.zero_grad()

            for step, batch in enumerate(tr_dl):
                B    = batch['input_ids'].size(0)
                ids  = batch['input_ids'].view(B*5, -1).to(DEVICE, non_blocking=True)
                mask = batch['attention_mask'].view(B*5, -1).to(DEVICE, non_blocking=True)
                correct = batch['correct'].to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                        enabled=(DEVICE.type == 'cuda')):
                    scores = model(ids, mask).view(B, 5)
                    loss   = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN']) / accum

                if torch.isfinite(loss):
                    if USE_SCALER:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()
                else:
                    print(f'    [warn] step {step}: non-finite loss, skipping backward')

                if (step + 1) % accum == 0:
                    if USE_SCALER:
                        scaler.unscale_(optim)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if USE_SCALER:
                        scaler.step(optim); scaler.update()
                    else:
                        optim.step()
                    sched.step()
                    optim.zero_grad()

                train_loss += loss.item() * B * accum; n_train += B

                if step % 25 == 0:
                    print(f'    step {step}/{len(tr_dl)}  '
                          f'loss={loss.item()*accum:.4f}  '
                          f'elapsed={time.time()-t_epoch:.0f}s', flush=True)

        
            model.eval()
            va_loss = 0.0; n_val = 0
            va_scores_all = []
            with torch.no_grad():
                for batch in va_dl:
                    B    = batch['input_ids'].size(0)
                    ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                    mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                    correct = batch['correct'].to(DEVICE)
                    with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                            enabled=(DEVICE.type == 'cuda')):
                        scores = model(ids, mask).view(B, 5)
                        loss   = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN'])
                    va_loss += loss.item() * B; n_val += B
                    va_scores_all.append(scores.cpu().float().numpy())
            va_scores_all = np.concatenate(va_scores_all)
            val_m = all_metrics(va_scores_all, y_idx[va_idx])

            tr_loss_val = train_loss / max(n_train, 1)
            va_loss_val = va_loss    / max(n_val,   1)

            print(f'  ep{epoch} ({time.time()-t_epoch:.0f}s)  '
                  f'train_loss={tr_loss_val:.4f} '
                  f'val_loss={va_loss_val:.4f} val_map3={val_m["map3"]:.4f} '
                  f'val_acc={val_m["acc"]:.4f} val_f1={val_m["f1"]:.4f}', flush=True)

     
            wlog({
                'ele/train_loss': tr_loss_val,
                'ele/val_loss':   va_loss_val,
                'ele/val_map3':   val_m['map3'],
                'ele/val_acc':    val_m['acc'],
                'ele/val_f1':     val_m['f1'],
                'ele/fold':       fold,
                'ele/epoch':      epoch,
            })

          
            if val_m['map3'] > best_val_map3:
                best_val_map3  = val_m['map3']
                best_va_scores = va_scores_all

               
                model.eval()
                te_scores = []
                with torch.no_grad():
                    for batch in te_dl:
                        B    = batch['input_ids'].size(0)
                        ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                        mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                        with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                                enabled=(DEVICE.type == 'cuda')):
                            scores = model(ids, mask).view(B, 5)
                        te_scores.append(scores.cpu().float().numpy())
                best_te_scores = np.concatenate(te_scores)

                best_state_dict = {k: v.detach().cpu().clone()
                                   for k, v in model.state_dict().items()}

        oof_scores[va_idx] = best_va_scores
        test_scores       += best_te_scores / n_folds

        save_path = os.path.join(save_dir, f'best_ele_fold{fold}.pth')
        torch.save(best_state_dict, save_path)
        print(f"  Saved best model, (val_map3={best_val_map3:.4f})", flush=True)

        wlog({'ele/best_val_map3': best_val_map3})
        wandb_finish()

        del model, best_state_dict; gc.collect(); torch.cuda.empty_cache()

    overall = all_metrics(oof_scores, y_idx)
    print(f'\n[ELECTRA] OOF MAP@3={overall["map3"]:.4f} '
          f'acc={overall["acc"]:.4f} f1={overall["f1"]:.4f}', flush=True)

   
    np.save(os.path.join(save_dir, 'test_ele.npy'), test_scores)
    np.save(os.path.join(save_dir, 'oof_ele.npy'),  oof_scores)
    np.save(f"{CFG['OUTPUT_DIR']}/test_ele.npy",    test_scores)
    np.save(f"{CFG['OUTPUT_DIR']}/oof_ele.npy",     oof_scores)
    print(f"Saved test_ele.npy + oof_ele.npy.")

    push_model(
        MODEL_TAG, save_dir,
        notes=f"v{RUN_VERSION}: {RUN_NOTES} | OOF MAP@3={overall['map3']:.4f} "
              f"acc={overall['acc']:.4f} f1={overall['f1']:.4f}"
    )

    return oof_scores, test_scores

print("train_electra_cv defined.")


train_electra_cv defined.


## Run Training

In [12]:
oof_ele = test_ele = None

if CFG['RUN_ELECTRA']:
    oof_ele, test_ele = train_electra_cv(train, test, n_folds=CFG['N_FOLDS'])
    print(f"\nTraining complete. test_ele shape={test_ele.shape}, oof_ele shape={oof_ele.shape}")
else:
    print("RUN_ELECTRA=False — training skipped.")


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



[ELECTRA] === fold 0 (train=1000, val=1000) ===


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 25839.10it/s]
[transformers] ElectraModel LOAD REPORT from: google/electra-large-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


    step 0/250  loss=0.4787  elapsed=1s
    step 25/250  loss=0.4986  elapsed=6s
    step 50/250  loss=0.4875  elapsed=12s
    step 75/250  loss=0.4995  elapsed=17s
    step 100/250  loss=0.4985  elapsed=23s
    step 125/250  loss=0.4660  elapsed=28s
    step 150/250  loss=0.4677  elapsed=34s
    step 175/250  loss=0.4765  elapsed=39s
    step 200/250  loss=0.4902  elapsed=44s
    step 225/250  loss=0.5403  elapsed=50s
  ep0 (69s)  train_loss=0.4840 val_loss=0.4573 val_map3=0.7015 val_acc=0.5580 val_f1=0.5586
    step 0/250  loss=0.5396  elapsed=0s
    step 25/250  loss=0.5028  elapsed=6s
    step 50/250  loss=0.5185  elapsed=11s
    step 75/250  loss=0.4655  elapsed=17s
    step 100/250  loss=0.4931  elapsed=22s
    step 125/250  loss=0.4326  elapsed=28s
    step 150/250  loss=0.3659  elapsed=33s
    step 175/250  loss=0.4767  elapsed=39s
    step 200/250  loss=0.3747  elapsed=44s
    step 225/250  loss=0.5019  elapsed=50s
  ep1 (69s)  train_loss=0.4538 val_loss=0.4413 val_map3=0.7458

ele/best_val_map3,▁
ele/epoch,▁█
ele/fold,▁▁
ele/train_loss,█▁
ele/val_acc,▁█
ele/val_f1,▁█
ele/val_loss,█▁
ele/val_map3,▁█
ele/best_val_map3,0.74583
ele/epoch,1
ele/fold,0


...


Uploading: 100%|██████████| 10.1k/10.1k [00:00<00:00, 24.1kB/s]

Upload successful: /teamspace/studios/this_studio/artifacts/test_ele.npy (10KB)
Starting upload for file /teamspace/studios/this_studio/artifacts/best_deb_fold1.pth



Uploading: 100%|██████████| 1.74G/1.74G [00:14<00:00, 116MB/s] 

Upload successful: /teamspace/studios/this_studio/artifacts/best_deb_fold1.pth (2GB)


Your model instance has been created.
Files are being processed...
See at: https://kaggle.com/models/kamal134134/smart-mcq-electra/pyTorch/updated_final_try
[hub] done. Inference notebooks can now kagglehub.model_download('kamal134134/smart-mcq-electra/pyTorch/updated_final_try').

Training complete. test_ele shape=(500, 5), oof_ele shape=(2000, 5)
